In [8]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [9]:
import yaml

with open("../config/base.yaml") as f:
    base = yaml.safe_load(f)

with open("../config/twibot22.yaml") as f:
    dataset = yaml.safe_load(f)

config = {**base, **dataset}

In [3]:
from src.data.loader import TwiBot22Loader
from src.data.preprocessor import TwiBot22Preprocessor

config["paths"]["raw_data"] = "../data/raw/"
loader = TwiBot22Loader(config)
data = loader.load_all()

preprocessor = TwiBot22Preprocessor(config)
processed = preprocessor.process_all(data)

Loading users...
Loaded 1000 users
Loading tweets (STREAMING mode)...
Streaming ../data/raw/TwiBot-22\tweet_0.json...
Stopped early at 2000 tweets (debug mode)
Loading edges...
Loaded 5000 edges
Loading labels...
Loaded 1000000 labels
Processing users...
Users processed: 1000
Processing tweets...
Tweets processed: 2000
Processing edges...
Edges processed: 5000
Processing labels...
Labels processed: 1000000


In [4]:
users_df = processed["users"]
tweets_df = processed["tweets"]
edges_df = processed["edges"]
labels_df = processed["labels"]

users_df.head()

,created_at,description,entities,id,location,name,pinned_tweet_id,profile_image_url,protected,public_metrics,url,username,verified,withheld
0,2020-01-16 02:02:55+00:00,Theoretical Computer Scientist. See also https...,"{'url': {'urls': [{'start': 0, 'end': 23, 'url...",u1217628182611927040,"Cambridge, MA",Boaz Barak,NaN,https://pbs.twimg.com/profile_images/125226236...,False,"{'followers_count': 7316, 'following_count': 2...",https://t.co/BoMip9FF17,boazbaraktcs,False,None
1,2014-07-02 17:56:46+00:00,creative _,None,u2664730894,🎈,olawale 💨,NaN,https://pbs.twimg.com/profile_images/147837638...,False,"{'followers_count': 123, 'following_count': 10...",,wale_io,False,None
2,2020-05-30 12:10:45+00:00,👽,None,u1266703520205549568,None,panagiota_.b,NaN,https://pbs.twimg.com/profile_images/142608606...,False,"{'followers_count': 3, 'following_count': 62, ...",,b_panagiota,False,None
3,2019-01-26 13:52:49+00:00,mama to maya. ABIM research pathway fellow @UV...,"{'description': {'mentions': [{'start': 43, 'e...",u1089159225148882949,"Charlottesville, VA","Jacqueline Hodges, MD MPH",NaN,https://pbs.twimg.com/profile_images/130229171...,False,"{'followers_count': 350, 'following_count': 57...",,jachodges_md,False,None
4,2009-04-30 19:01:42+00:00,Father / SWT Alumnus / Longhorn Fan,None,u36741729,United States,Matthew Stubblefield,NaN,https://pbs.twimg.com/profile_images/145808462...,True,"{'followers_count': 240, 'following_count': 29...",,Matthew_Brody,False,None


In [5]:
tweets_df.head()
edges_df.head()
labels_df.head()

,id,label
0,u1217628182611927040,human
1,u2664730894,human
2,u1266703520205549568,human
3,u1089159225148882949,human
4,u36741729,bot


In [6]:
from src.data.entity_extractor import EntityExtractor

extractor = EntityExtractor(config)
entities = extractor.extract_all(processed["tweets"])

entities

Extracting hashtags...
Hashtags extracted: 324
Tweet-Hashtag edges: 604
Extracting URLs...
URLs extracted: 118
Tweet-URL edges: 600


{'hashtags':                     hashtag
 0          malikfaisalakram
 1    expelrussianambassador
 2       endpoliceharassment
 3              mymediaprime
 4           allforteammelli
 ..                      ...
 319    mytwitteranniversary
 320         राजीवलोचन_मंदिर
 321                    ブレスロ
 322                   コシュニエ
 323        digitalsherlocks
 
 [324 rows x 1 columns],
 'hashtag_edges':                  tweet_id                hashtag
 0    t1495109640459198464  thelegendofvoxmachina
 1    t1494529099049496578   criticalrolespoilers
 2    t1494487809507295237            lvmspoilers
 3    t1494475376495509504            lvmspoilers
 4    t1493872834229145600                    wip
 ..                    ...                    ...
 599  t1479657962654388226                covid19
 600  t1479657962654388226                   misc
 601  t1476748966930886656                pedsicu
 602  t1476746667378548741                pedsicu
 603  t1468751811754532864                peds

In [7]:
from src.data.graph_builder import GraphBuilder

builder = GraphBuilder(config)
graph = builder.build_graph(processed, entities)

graph

Building Heterogeneous Graph...
Building ID maps...
Node counts:
HeteroData(
  user={ num_nodes=1000 },
  tweet={ num_nodes=2000 },
  hashtag={ num_nodes=324 },
  url={ num_nodes=118 }
)
Processing labels...
Graph construction complete.


HeteroData(
  user={
    num_nodes=1000,
    y=[1000]
  },
  tweet={ num_nodes=2000 },
  hashtag={ num_nodes=324 },
  url={ num_nodes=118 },
  (user, follows, user)={ edge_index=[2, 6] },
  (user, posts, tweet)={ edge_index=[2, 0] },
  (tweet, contains, hashtag)={ edge_index=[2, 604] },
  (tweet, links, url)={ edge_index=[2, 600] }
)